# Magnetic Resonance Imaging II

## Exercise 2

---

**Prof. Dr. Markus Zimmermann**

**Mohamed Wassim Bensalem**

---

TODO

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import sys
from typing import Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

# Lokale Module importieren
sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(
    0, "/home/jovyan/shared-folder/Grundlagen-der-tomographischen-Bildgebung/mri/src"
)

from typing import Optional, Tuple

import torch
from mri.conopt.linops import LinearOperator
from mri.parim.analytic import cyclic_head_coil
from tqdm import tqdm

## System and Sequence Parameters

TODO

In [ ]:
seq_filename = "gre.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128

case = ["T1-weighted", "T2-weighted", "PD-weighted"][0]

if case == "T1-weighted":
    flip_angle_deg = 90
    tr = 250e-3
    te = 5e-3
elif case == "T2-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 50e-3
elif case == "PD-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 5e-3

torch.set_default_device("cpu")

## System and Sequence Object

TODO

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Task 1: Sequence components

TODO

Copy from last exercise

In [ ]:
# Your solution here

delta_kx = 1 / fov_x
delta_ky = 1 / fov_y
raster_eps = 1e-12

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=gx.flat_time,
    delay=gx.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx)
    - pp.calc_duration(gx_spoiler)
    - te_delay
)

te_delay = (
    np.ceil((te_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)
tr_delay = (
    np.ceil((tr_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)

## Task 2: Sequence Assembly

TODO

Copy from last exercise but change from

```python
phase_areas = np.linspace(-n_y / 2, n_y / 2, num=n_y, endpoint=False) * delta_ky
```

to

```python
phase_areas = np.linspace(-n_y / 2, n_y / 2, num=n_y // 2, endpoint=False) * delta_ky
```

Thereby skipping every other line (acceleration factor 2)

In [ ]:
# Your solution here

phase_areas = np.linspace(-n_y / 2, n_y / 2, num=n_y // 2, endpoint=False) * delta_ky

for phase_area in phase_areas:

    seq.add_block(rf)

    gy_pre = pp.make_trapezoid(
        channel="y",
        area=phase_area,
        duration=pp.calc_duration(gx_pre),
        system=system,
    )

    seq.add_block(gx_pre, gy_pre)

    seq.add_block(pp.make_delay(te_delay))

    seq.add_block(gx, adc)

    seq.add_block(gx_spoiler)

    seq.add_block(pp.make_delay(tr_delay))

## Verification and Export
 
TODO

Run code

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot(time_range=(0.0, tr - tr_delay + te_delay))

seq.write(seq_filename)

## Phantom and Simulation
 
TODO

New here is 

```python
num_channels = 8
coil_sensitivities = cyclic_head_coil((num_channels, n_x, n_y, 1))
phantom.coil_sens = coil_sensitivities.to(torch.complex64)
```

Maybe run code?

In [ ]:
phantom = mr0.util.load_phantom(size=(n_x, n_y))

num_channels = 8
coil_sensitivities = cyclic_head_coil((num_channels, n_x, n_y, 1))
phantom.coil_sens = coil_sensitivities.to(torch.complex64)

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Task 3: Reconstruction and Display

TBD

Write `fourier_transform_forward` and `fourier_transform_adjoint` function

Perform simple zero-filled reconstruction

In [ ]:
def fourier_transform_forward(
    x: torch.Tensor,
    dim: Tuple[int],
) -> torch.Tensor:

    x = torch.fft.ifftshift(x, dim=dim)
    x = torch.fft.ifftn(x, dim=dim)
    x = torch.fft.fftshift(x, dim=dim)

    return x


def fourier_transform_adjoint(
    x: torch.Tensor,
    dim: Tuple[int],
) -> torch.Tensor:

    x = torch.fft.ifftshift(x, dim=dim)
    x = torch.fft.ifftn(x, dim=dim)
    x = torch.fft.fftshift(x, dim=dim)

    return x

In [ ]:
# Your solution here

data = signal.clone()

data = data.reshape((n_y // 2, n_x, num_channels)).permute((2, 1, 0))

data_full = torch.zeros((num_channels, n_x, n_y), dtype=data.dtype)
data_full[:, :, ::2] = data

coil_images = fourier_transform_adjoint(data_full, (1, 2))

plt.figure()
for i in range(num_channels):
    plt.subplot(2, 4, i+1)
    plt.imshow(np.abs(coil_images[i, :, :]), cmap="gray")
    plt.title(f"Channel {i+1}")
plt.show()

## Task 4-: Operator Formalism

- Fourier Transform Operator
- ChannelOperator
- DiagonalOperator (Masking)

In [ ]:
class FourierTransformOperator(LinearOperator):
    """
    Applies the standard Fourier transform along specified axes.
    """

    def __init__(
        self,
        input_shape: Tuple[int, ...],
        axes: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        """
        Initialize the FourierTransformOperator.

        Parameters:
        ----------
        input_shape : tuple of int
            Shape of the input tensor.
        axes : tuple of int
            Axes along which to perform the Fourier transform.
        """
        self.shape = (
            torch.prod(torch.tensor(input_shape)),
            torch.prod(torch.tensor(input_shape)),
        )
        self.forward_shape = input_shape
        self.adjoint_shape = input_shape
        self.axes = axes

        # Set the data type to float32 and internal dtype to complex64
        self.dtype = torch.complex64
        self.device = device

    def _matvec(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        """
        Perform the forward Fourier transform.

        Parameters:
        ----------
        input_tensor : torch.Tensor
            Input tensor to be transformed.

        Returns:
        -------
        torch.Tensor
            Transformed tensor after applying the Fourier transform.
        """
        input_tensor = input_tensor.reshape(self.forward_shape)
        output_tensor = fourier_transform_forward(input_tensor, self.axes)
        return output_tensor.ravel()

    def _rmatvec(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        """
        Perform the adjoint (inverse) Fourier transform.

        Parameters:
        ----------
        input_tensor : torch.Tensor
            Input tensor to apply the adjoint (inverse) Fourier transform.

        Returns:
        -------
        torch.Tensor
            The result after applying the inverse Fourier transform.
        """
        input_tensor = input_tensor.reshape(self.adjoint_shape)
        output_tensor = fourier_transform_adjoint(input_tensor, self.axes)
        return output_tensor.ravel()

In [ ]:
class ChannelOperator(LinearOperator):
    """
    A linear operator that applies forward and adjoint operations using coil sensitivity
    maps.
    """

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        """
        Initialize the ChannelOperator.

        Parameters:
        ----------
        coil_sensitivities : torch.Tensor
            Coil sensitivity maps (nC, nX, nY, nZ).
        data_shape : tuple of int
            Shape of the data (nX, nY, nZ, nS, nTI, nTE).
        normalize : bool, optional
            Whether to normalize to unit spectral norm (default is True).
        device : str, optional
            Device on which the operator will run ('cpu' or 'cuda').
        """
        # Unpack the data shape
        nX, nY = data_shape

        # Number of channels (coils)
        nC = coil_sensitivities.shape[0]

        # Device and dtype
        self.device = device
        self.dtype = torch.complex64

        # Define forward and adjoint shapes
        self.forward_shape = (1, nX, nY)
        self.adjoint_shape = (nC, nX, nY)

        # Define the shape for complex-to-real operations
        self.shape = (
            torch.prod(torch.tensor(self.adjoint_shape)),
            torch.prod(torch.tensor(self.forward_shape)),
        )

        # Store coil sensitivities, ensuring correct dimensions and device
        self.coil_sensitivities = coil_sensitivities

    def _matvec(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        """
        Apply the forward channel operator (coil sensitivities).

        This operation multiplies the input tensor by the coil sensitivity maps.

        Parameters:
        ----------
        input_tensor : torch.Tensor
            Input data tensor to be transformed.

        Returns:
        -------
        torch.Tensor
            The result after applying the forward channel operator.
        """
        # Reshape the input data to the appropriate internal dtype and shape
        input_tensor = input_tensor.reshape(self.forward_shape)

        # Apply the coil sensitivities
        output_tensor = self.coil_sensitivities * input_tensor

        # Flatten the result and cast back to the external dtype
        output_tensor = output_tensor.ravel()

        return output_tensor

    def _rmatvec(
        self,
        coil_data: torch.Tensor,
    ) -> torch.Tensor:
        """
        Apply the adjoint channel operator (combining coil signals).

        This operation sums the input data weighted by the conjugate of the coil
        sensitivity maps.

        Parameters:
        ----------
        coil_data : torch.Tensor
            Input data tensor representing the coil signals.

        Returns:
        -------
        torch.Tensor
            The result after applying the adjoint channel operator.
        """
        # Reshape the input data to the appropriate internal dtype and shape
        coil_data = coil_data.reshape(self.adjoint_shape)

        # Apply the adjoint (combining the signals using the conjugate of the
        # sensitivities)
        output_tensor = torch.sum(
            torch.conj(self.coil_sensitivities) * coil_data, dim=0
        )

        # Flatten the result and cast back to the external dtype
        output_tensor = output_tensor.ravel()

        return output_tensor

In [ ]:
class DiagnonalOperator(LinearOperator):
    """
    A linear operator that applies forward and adjoint operations using coil sensitivity
    maps.
    """

    def __init__(
        self,
        diagonal_tensor: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        """
        Initialize the ChannelOperator.

        Parameters:
        ----------
        coil_sensitivities : torch.Tensor
            Coil sensitivity maps (nC, nX, nY, nZ).
        data_shape : tuple of int
            Shape of the data (nX, nY, nZ, nS, nTI, nTE).
        normalize : bool, optional
            Whether to normalize to unit spectral norm (default is True).
        device : str, optional
            Device on which the operator will run ('cpu' or 'cuda').
        """
        # Unpack the data shape
        num_channels, nX, nY = data_shape

        # Device and dtype
        self.device = device
        self.dtype = torch.complex64

        # Define forward and adjoint shapes
        self.forward_shape = (num_channels, nX, nY)
        self.adjoint_shape = (num_channels, nX, nY)

        # Define the shape for complex-to-real operations
        self.shape = (
            torch.prod(torch.tensor(self.adjoint_shape)),
            torch.prod(torch.tensor(self.forward_shape)),
        )

        # Store coil sensitivities, ensuring correct dimensions and device
        self.diagonal_tensor = diagonal_tensor

    def _matvec(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        """
        Apply the forward channel operator (coil sensitivities).

        This operation multiplies the input tensor by the coil sensitivity maps.

        Parameters:
        ----------
        input_tensor : torch.Tensor
            Input data tensor to be transformed.

        Returns:
        -------
        torch.Tensor
            The result after applying the forward channel operator.
        """
        # Reshape the input data to the appropriate internal dtype and shape
        input_tensor = input_tensor.reshape(self.forward_shape)

        # Apply the coil sensitivities
        output_tensor = self.diagonal_tensor * input_tensor

        # Flatten the result and cast back to the external dtype
        output_tensor = output_tensor.ravel()

        return output_tensor

    def _rmatvec(
        self,
        input_tensor: torch.Tensor,
    ) -> torch.Tensor:
        """
        Apply the adjoint channel operator (combining coil signals).

        This operation sums the input data weighted by the conjugate of the coil
        sensitivity maps.

        Parameters:
        ----------
        coil_data : torch.Tensor
            Input data tensor representing the coil signals.

        Returns:
        -------
        torch.Tensor
            The result after applying the adjoint channel operator.
        """
        # Reshape the input data to the appropriate internal dtype and shape
        input_tensor = input_tensor.reshape(self.adjoint_shape)

        # Apply the adjoint (combining the signals using the conjugate of the
        # sensitivities)
        output_tensor = torch.conj(self.diagonal_tensor) * input_tensor

        # Flatten the result and cast back to the external dtype
        output_tensor = output_tensor.ravel()

        return output_tensor

## Create Operators

TBD

In [ ]:
mask = torch.zeros((num_channels, n_x, n_y), dtype=torch.complex64)
mask[:, :, ::2] = 1

F = FourierTransformOperator((num_channels, n_x, n_y), (1, 2))
C = ChannelOperator(coil_sensitivities[..., 0].to(torch.complex64), (n_x, n_y))
M = DiagnonalOperator(mask, (num_channels, n_x, n_y))

## Zero-filled recon with Operators

TBD

In [ ]:
data_full_vec = data_full.ravel()

In [ ]:
AHd = C.H @ F.H @ M.H @ data_full_vec

In [ ]:
AHd_image = AHd.reshape((n_x, n_y))

In [ ]:
plt.figure()
plt.imshow(np.abs(AHd_image), cmap="gray")
plt.title(f"Zero-filled GRE Image — {case}")
plt.show()

## Iterative recon with Conjugate Gradient Method

TBD

In [ ]:
def conjugate_gradient(
    A: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter)):

        # Compute gradient
        g = A @ x - b

        # Conjugacy factor: mix of previous direction to keep search directions A-conjugate
        conjugacy_factor = g.conj() @ g / (g_prev.conj() @ g_prev)

        # Update search direction
        p = g + conjugacy_factor * p_prev

        # Compute stepsize
        stepsize = g.conj() @ g / (p.conj() @ (A @ p))

        # Update x
        x = x - stepsize * p

        # Store current gradient for next iteration
        g_prev = g.clone()
        p_prev = p.clone()

    return x

In [ ]:
AHA = C.H @ F.H @ M @ F @ C
AHd = C.H @ F.H @ M @ data_full_vec
x = torch.zeros((n_x, n_y), dtype=torch.complex64).ravel()

In [ ]:
x_recon = conjugate_gradient(
    AHA,
    AHd,
    x,
    maxiter=30,
)

In [ ]:
x_recon_image = x_recon.reshape((n_x, n_y))

## Comparison

TBD

In [ ]:
plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(np.abs(AHd_image), cmap="gray")
plt.title(f"Zerofilled GRE Image — {case}")
plt.subplot(1, 2, 2)
plt.imshow(np.abs(x_recon_image), cmap="gray")
plt.title(f"Reconstructed GRE Image — {case}")
plt.show()